In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

# Các hàm phụ trợ để lấy thông tin chi tiết từ ID nhận được từ API
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

def getPayloadData(data):
    for x in data['payloads']:
       for y in x:
        if y:
            response = requests.get("https://api.spacexdata.com/v4/payloads/"+str(y)).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

def getCoreData(data):
    for x in data['cores']:
                for core in x:
                    if core['core'] is not None:
                        response = requests.get("https://api.spacexdata.com/v4/cores/"+str(core['core'])).json()
                        Block.append(response['block'])
                        ReusedCount.append(response['reuse_count'])
                        Serial.append(response['serial'])
                    else:
                        Block.append(None)
                        ReusedCount.append(None)
                        Serial.append(None)
                    Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
                    Flights.append(core['flight'])
                    GridFins.append(core['gridfins'])
                    Reused.append(core['reused'])
                    Legs.append(core['legs'])
                    LandingPad.append(core['landpad'])

# 1. Gọi API lấy dữ liệu thô các lần phóng trong quá khứ
spacex_url="https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
data = pd.json_normalize(response.json())

# Lọc các cột cần thiết và định dạng ngày tháng
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

# Khai báo các danh sách để lưu dữ liệu sau khi gọi các hàm phụ trợ
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Longitude = []
Latitude = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []

# Gọi các hàm để điền thông tin chi tiết vào danh sách
getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)

# Tạo dictionary chứa dữ liệu sạch
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# 2. Tạo DataFrame
df = pd.DataFrame(launch_dict)

# Lọc chỉ giữ lại các lần phóng của Falcon 9
df_falcon9 = df[df['BoosterVersion']=='Falcon 9'].copy()
df_falcon9.loc[:, 'FlightNumber'] = list(range(1, df_falcon9.shape[0] + 1))

# 3. Xử lý giá trị bị khuyết (NaN) ở cột PayloadMass bằng giá trị trung bình (mean)
mean_payload = df_falcon9['PayloadMass'].mean()
df_falcon9['PayloadMass'].fillna(mean_payload, inplace=True)

# 4. Xuất dữ liệu ra file CSV
df_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Thu thập dữ liệu API hoàn tất! File 'dataset_part_1.csv' đã được lưu.")

Thu thập dữ liệu API hoàn tất! File 'dataset_part_1.csv' đã được lưu.


C:\Users\84333\AppData\Local\Temp\ipykernel_11072\1937517020.py:113: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_falcon9['PayloadMass'].fillna(mean_payload, inplace=True)
